# Introductory example

In this notebook, we compute income taxes and social security contributions for example
data.

In [ ]:
import pandas as pd

from gettsim import InputData, MainTarget, TTTargets, copy_environment, main, tt

## Creating the Data

The first step in GETTSIM's workflow is to define the targets of the taxes and transfers
system you are interested in. The key sequences of the nested dictionary below are the
paths GETTSIM will use as targets. For instance, via the path `einkommensteuer` and
`betrag_m_sn`, we request the amount of income tax to be paid monthly at the
Steuernummer level. *Note: Of course, the income tax is paid annually and calculated at
that level, but GETTSIM will do the conversion for you.*

The values on the lowest level of the dictionaries (called leaves) will be used as the
column names of the resulting DataFrame. Here, `income_tax_m` will be the name of the
column containing the income tax results.

In this example, we are interested in the income tax and the social insurance
contributions paid when being in regular employment.

In [ ]:
TT_TARGETS = {
    "einkommensteuer": {"betrag_m_sn": "income_tax_m"},
    "sozialversicherung": {
        "pflege": {
            "beitrag": {
                "betrag_versicherter_m": "long_term_care_insurance_contribution_m"
            }
        },
        "kranken": {
            "beitrag": {"betrag_versicherter_m": "health_insurance_contribution_m"}
        },
        "rente": {
            "beitrag": {"betrag_versicherter_m": "pension_insurance_contribution_m"}
        },
        "arbeitslosen": {
            "beitrag": {
                "betrag_versicherter_m": "unemployment_insurance_contribution_m"
            }
        },
    },
}

Next, we need to find out which input data we actually need to calculate the targets we
are interested in. We can do this by specifying a template as the `main_target` of
`gettsim.main`. The template returns the input variables needed to compute the specified
`tt_targets`.

Some of these inputs are computed from other inputs. If you already know the value of
such a computed input, you can provide it directly in the template call. GETTSIM will
then exclude its upstream dependencies from the template, giving you a shorter list of
remaining inputs to fill. For example, the old-age pension benefit
(`sozialversicherung__rente__altersrente__betrag_m`) depends on many pension-related
inputs (entitlement points, contribution months, etc.). Since nobody in our scenario is
retired, we provide it as 0, which removes all of those upstream inputs from the
template.

In [ ]:
main(
    main_target=MainTarget.templates.input_data_dtypes.tree,
    policy_date_str="2025-01-01",
    tt_targets=TTTargets.tree(TT_TARGETS),
    input_data=InputData.tree(
        {
            "p_id": pd.Series([0]),
            "sozialversicherung": {
                "rente": {
                    "altersrente": {"betrag_m": pd.Series([0])},
                },
            },
        }
    ),
    include_warn_nodes=False,
)

The output above is a nested dictionary whose leaves are dtype hints. Each leaf
corresponds to an input variable that GETTSIM needs. To build the mapper (below), we
replace each dtype hint with a column name from our input DataFrame.

Now, we create some example data. Our example household consists of a married couple
(both 30 years old, both employed) with a 10-year-old child. Here, we use a pandas
DataFrame with column names that are different from the ones GETTSIM expects.

In [ ]:
DATA = pd.DataFrame(
    {
        "age": [30, 30, 10],
        "working_hours": [35, 35, 0],
        "disability_grade": [0, 0, 0],
        "birth_year": [1995, 1995, 2015],
        "hh_id": [0, 0, 0],
        "p_id": [0, 1, 2],
        "self_employed": [False, False, False],
        "income_from_self_employment": [0, 0, 0],
        "income_from_rent": [0, 0, 0],
        "income_from_employment": [5000, 4000, 0],
        "income_from_forest_and_agriculture": [0, 0, 0],
        "income_from_capital": [500, 0, 0],
        "income_from_other_sources": [0, 0, 0],
        "contribution_to_private_pension_insurance": [0, 0, 0],
        "childcare_expenses": [0, 0, 0],
        "person_that_pays_childcare_expenses": [-1, -1, 0],
        "joint_taxation": [True, True, False],
        "contribution_private_health_insurance": [0, 0, 0],
        "has_children": [True, True, False],
        "single_parent": [False, False, False],
        "is_child": [False, False, True],
        "spouse_id": [1, 0, -1],
        "parent_id_1": [-1, -1, 0],
        "parent_id_2": [-1, -1, 1],
        "in_training": [False, False, False],
        "id_recipient_child_allowance": [-1, -1, 0],
    }
)

Next, we define a mapping from GETTSIM's expected input structure to our data. At each
leaf, we either put a column name from `DATA` or a constant value.

In [ ]:
MAPPER = {
    "alter": "age",
    "arbeitsstunden_w": "working_hours",
    "behinderungsgrad": "disability_grade",
    "geburtsjahr": "birth_year",
    "hh_id": "hh_id",
    "p_id": "p_id",
    "einnahmen": {
        "bruttolohn_m": "income_from_employment",
        "kapitalerträge_m": "income_from_capital",
        "renten": {
            "betriebliche_altersvorsorge_m": 0.0,
            "geförderte_private_vorsorge_m": 0.0,
            "sonstige_private_vorsorge_m": 0.0,
            "aus_berufsständischen_versicherungen_m": 0.0,
        },
    },
    "einkommensteuer": {
        "einkünfte": {
            "ist_hauptberuflich_selbstständig": "self_employed",
            "aus_gewerbebetrieb": {"betrag_m": "income_from_self_employment"},
            "aus_vermietung_und_verpachtung": {"betrag_m": "income_from_rent"},
            "aus_forst_und_landwirtschaft": {
                "betrag_m": "income_from_forest_and_agriculture"
            },
            "aus_selbstständiger_arbeit": {"betrag_m": "income_from_self_employment"},
            "sonstige": {
                "alle_weiteren_m": "income_from_other_sources",
                "rente": {
                    "alter_beginn_leistungsbezug_sonstige_private_vorsorge": 65,
                },
            },
        },
        "abzüge": {
            "beitrag_private_rentenversicherung_m": (
                "contribution_to_private_pension_insurance"
            ),
            "kinderbetreuungskosten_m": "childcare_expenses",
            "p_id_kinderbetreuungskostenträger": "person_that_pays_childcare_expenses",
        },
        "gemeinsam_veranlagt": "joint_taxation",
    },
    "sozialversicherung": {
        "rente": {
            "jahr_renteneintritt": 2080,
            "altersrente": {
                "betrag_m": 0.0,
            },
            "erwerbsminderung": {
                "betrag_m": 0.0,
            },
        },
        "kranken": {
            "beitrag": {"privat_versichert": "contribution_private_health_insurance"}
        },
        "pflege": {"beitrag": {"hat_kinder": "has_children"}},
    },
    "familie": {
        "alleinerziehend": "single_parent",
        "kind": "is_child",
        "p_id_ehepartner": "spouse_id",
        "p_id_elternteil_1": "parent_id_1",
        "p_id_elternteil_2": "parent_id_2",
    },
    "kindergeld": {
        "in_ausbildung": "in_training",
        "p_id_empfänger": "id_recipient_child_allowance",
    },
}

In practice, you would probably save the template to disk (e.g. as a YAML file), edit
the leaves there, and read it back in as the mapper. Remember to allow for unicode
characters, since many variable names contain Umlaute.

```python
import yaml

with PATH_FOR_TEMPLATE.open("w") as f:
    yaml.dump(TEMPLATE, f, allow_unicode=True)

# Edit the leaves in the template, then read it back in
with PATH_FOR_TEMPLATE.open("r") as f:
    MAPPER = yaml.safe_load(f)
```

Some inputs may not be directly relevant to the scenario at hand. For example,
`jahr_renteneintritt` and
`alter_beginn_leistungsbezug_sonstige_private_vorsorge` only matter for people who
actually receive pensions. Because GETTSIM's DAG is static, these inputs are still
required even when the corresponding benefit is zero. In such cases, assign a reasonable
default value — the exact number does not matter (as long as the benefit itself is zero),
but it must be a valid input (e.g. a plausible year, not 0 or `None`).

## Calculating taxes and transfers

GETTSIM's `main` function is powered by a DAG. This has several advantages:
- You can select any part of the DAG as a target, giving access to intermediate results.
- You can feed any part of the DAG as input, overwriting specific parts (e.g. the
  policy environment).
- You can skip parts of the DAG (e.g. safety checks on input data) to speed up
  computation, at the expense of less informative error messages.

First, we compute the targets defined above using the input data. In a second example,
we manipulate the policy environment to see why the interface DAG is useful.

### Simple computation

Let's calculate taxes and transfers first:

In [ ]:
result = main(
    policy_date_str="2025-01-01",
    input_data=InputData.df_and_mapper(
        df=DATA,
        mapper=MAPPER,
    ),
    main_target=MainTarget.results.df_with_mapper,
    tt_targets=TTTargets.tree(TT_TARGETS),
    include_warn_nodes=False,
)
result.T

### Manipulating the policy environment

First, we obtain the policy environment for the policy date we're interested in. Similar
to above, we call the `main` function.

In [ ]:
status_quo = main(
    policy_date_str="2025-01-01",
    main_target=MainTarget.policy_environment,
)

Let us modify the policy environment by increasing the contribution rate of the public
pension insurance by 1 percentage point. 

The first step is to create a copy.

In [ ]:
increased_rate = copy_environment(status_quo)


The contribution rate is a `ScalarParam` object:

In [ ]:
type(status_quo["sozialversicherung"]["rente"]["beitrag"]["beitragssatz"])

We get the current `value` of the `ScalarParam` out. We then inject a new `ScalarParam` object into the same place of `policy_environment`:

In [ ]:
old_beitragssatz = status_quo["sozialversicherung"]["rente"]["beitrag"]["beitragssatz"]
increased_rate["sozialversicherung"]["rente"]["beitrag"]["beitragssatz"] = (
    tt.ScalarParam(value=old_beitragssatz.value + 0.01)
)

Now we can compute taxes and transfers with the increased contribution rate:

In [ ]:
result = main(
    main_target=MainTarget.results.df_with_mapper,
    policy_date_str="2025-01-01",
    policy_environment=increased_rate,
    input_data=InputData.df_and_mapper(
        df=DATA,
        mapper=MAPPER,
    ),
    tt_targets=TTTargets.tree(TT_TARGETS),
    include_warn_nodes=False,
)
result.T